## 기본구조 2

In [ ]:
# MessagesPlaceholder를 사용하면, 시스템 메시지와 현재 사용자 입력 사이에 이전 대화 기록 전체를 유연하게 삽입할 수 있어서, 상태(State)를 가진 대화형 애플리케이션을 만들 때 필수적입니다.

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder  # 핵심!
)
from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

# 1. 이전 대화 기록 (History) 예시
# 이는 실제 애플리케이션에서는 memory components(기억저장자치) 등에서 가져옵니다.
chat_history = [
    HumanMessage(content="안녕하세요! 저는 파이썬 개발자입니다."),
    AIMessage(content="반갑습니다. 파이썬 개발에 대해 어떤 것을 도와드릴까요?"),
]

# 2. ChatPromptTemplate 정의
# from_messages를 사용하여 템플릿의 구조를 정의합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        # 1) 시스템 지침 (System Instruction)
        # 챗봇의 역할을 정의합니다.
        SystemMessagePromptTemplate.from_template(
            "당신은 친절하고 전문적인 AI 어시스턴트입니다. 모든 답변은 한국어로 작성해 주세요."
        ),

        # 2) 대화 기록 Placeholder (MessagesPlaceholder:LLM 챗봇 만들 때 대화구조 정의하고 맥락을 구분하기 위해 사용되는 표식임)
        # 이 부분이 동적으로 'chat_history'가 들어갈 위치를 지정합니다.
        # LLM 에게 메세지를 명확히 구분할 수 있도록 한다. 대화 기록들이 일관적으로 쌓일 수 있게끔 구조화하는 역할을 한다.-> 맥락 유지에 도움
        # variable_name을 통해 나중에 format 함수에 어떤 이름으로 전달할지 지정합니다.
        MessagesPlaceholder(variable_name="history"),

        # 3) 사용자의 새로운 입력 (Current Human Input)
        # 현재 사용자가 모델에게 보내는 질문입니다.
        HumanMessagePromptTemplate.from_template("{input}")
    ]
)

# 3. Prompt Template에 실제 값 바인딩 및 최종 PromptValue 생성
# prompt.invoke() 또는 prompt.format_prompt()를 사용하여 값을 채웁니다.
final_prompt_value = prompt.invoke(
    {
        "history": chat_history,  # MessagesPlaceholder(variable_name="history")에 바인딩(템플릿 안의 변수 자리에 실제 값을 연결해서 넣어준다)
        "input": "파이썬으로 웹 크롤링을 할 때 가장 많이 사용하는 라이브러리는 무엇인가요?" # HumanMessagePromptTemplate에 바인딩
    }
)

# 4. 결과 확인
print("--- 최종 PromptValue 내용 (Messages) ---")
print(final_prompt_value.messages)

print("\n--- 각 메시지 유형 확인 ---")
for message in final_prompt_value.messages:
    print(f"[{message.type.capitalize()}]: {message.content[:50]}...")

Prompt 예시

  PromptTemplate:

  문자열 하나를 만든다

  ChatPromptTemplate:

  system / human / ai 같은 역할이 있는 메시지 목록을 만든다


  - PromptTemplate: 그냥 긴 문자열 프롬프트를 만들 때
  - ChatPromptTemplate: system/human 역할을 나눠서 대화형 프롬프트를 만들 때
  - 둘 다 {context} 같은 변수를 바인딩해서 사용할 수 있음
  - 요약, 번역, RAG 같은 단일 작업에는 PromptTemplate도 자주 씀
> 만약, 대화흐름이 중요하면 ChatPromptTemplate을 쓰는 게 더 적합하다. 역할 구분과 이전 대화 기록이 필요한 경우니까! 물론, PromptTemplate로 억지로 만들 수는 있다.

정리  
  - 요약, 번역, 분류, 추출, 단발성 질문 → PromptTemplate
  - 챗봇, 튜터, 상담, 이전 대화 기억이 필요한 작업 → ChatPromptTemplate

In [ ]:
# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt_template = """Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:"
"""

prompt = PromptTemplate.from_template(prompt_template)
prompt

# 이 코드는 일반 문자열 프롬프트 템플릿 방식이다. 즉, context 자리에 실제 문서를 넣어서 하나의 긴 문자열 프롬프트를 만드는 방식이다. 
# 다른 점은? 지금까지 살펴본 ChatPromptTemplate 방식은 역할이 있는 메시지 구조이다. 

In [ ]:
# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt_template = """You are a helpful expert journalist in extracting the main themes from a GIVEN DOCUMENTS below.
Please provide a comprehensive summary of the GIVEN DOCUMENTS in numbered list format. 
The summary should cover all the key points and main ideas presented in the original text, while also condensing the information into a concise and easy-to-understand format. 
Please ensure that the summary includes relevant details and examples that support the main ideas, while avoiding any unnecessary information or repetition. 
The length of the summary should be appropriate for the length and complexity of the original text, providing a clear and accurate overview without omitting any important information.

GIVEN DOCUMENTS:
{docs}

FORMAT:
1. main theme 1
2. main theme 2
3. main theme 3
...

CAUTION:
- DO NOT list more than 5 main themes.

Helpful Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)
prompt
# 내가 LLM에게서 하고자 하는 것을 프롬프트에 잘 녹여내는 것이 중요하다. 

In [ ]:
# 실제 사용할 때에는 llm과 체이닝 하여 사용할 수 있다. 이런식으로!

prompt = PromptTemplate.from_template(prompt_template)

chain = prompt | llm

response = chain.invoke({
    "context": "코너 맥그리거"
})
print(response.content)